# Breast Cancer Screening: An Explainable ML Demonstration

**YBI Foundation | 45-Day Machine Learning & Data Science Internship**

> **Safety note:** This notebook is an educational demonstration. It is not a diagnostic or clinical decision-support tool.

## 1. Problem statement

Can diagnostic measurements be used to estimate a positive malignant-screen label? We build a reproducible binary classifier and evaluate it with metrics appropriate for a screening-style problem. The aim is learning responsible machine-learning workflow, not medical deployment.

In [1]:
import sys
from pathlib import Path

# Makes the notebook work from its notebooks/ folder or in Colab after cloning.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay
from src.train import load_data, build_model, RANDOM_STATE

sns.set_theme(style='whitegrid')

Imports complete.\n

## 2. Data understanding

The scikit-learn Wisconsin Diagnostic Breast Cancer dataset contains 569 rows and 30 numeric diagnostic measurements. We recode the target so `1` represents a malignant (positive) screen.

In [2]:
X, y = load_data()
print(f'Rows: {X.shape[0]} | Features: {X.shape[1]}')
print('\nTarget distribution (0 = benign screen, 1 = positive malignant screen):')
print(y.value_counts().sort_index())
X.head()

Rows: 569 | Features: 30\n\nTarget distribution (0 = benign screen, 1 = positive malignant screen):\npositive_screen\n0    357\n1    212\nName: count, dtype: int64\n

## 3. Split strategy and preprocessing

A stratified 75/25 split preserves the class balance. Scaling is placed **inside** a pipeline, so it is fitted only on the training data and cannot leak information from the test set.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
model = build_model()
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.50).astype(int)
print(f'Training rows: {len(X_train)} | Test rows: {len(X_test)}')

Training rows: 426 | Test rows: 143\n

## 4. Evaluation

For a screening context, accuracy alone can hide important errors. We report ROC-AUC and average precision for ranking quality, then sensitivity and specificity at the chosen 0.50 threshold.

In [4]:
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, confusion_matrix
cm = confusion_matrix(y_test, pred)
metrics = pd.Series({
    'Accuracy': accuracy_score(y_test, pred),
    'ROC-AUC': roc_auc_score(y_test, proba),
    'Average precision': average_precision_score(y_test, proba),
    'Sensitivity': cm[1, 1] / cm[1].sum(),
    'Specificity': cm[0, 0] / cm[0].sum(),
}).round(3)
print(metrics.to_string())
ConfusionMatrixDisplay(cm, display_labels=['Benign screen', 'Positive screen']).plot(cmap='Blues')
plt.title('Test-set confusion matrix')
plt.show()

Accuracy             0.958\nROC-AUC              0.995\nAverage precision    0.991\nSensitivity          0.962\nSpecificity          0.956\n

## 5. Interpretation

Permutation importance measures the drop in test ROC-AUC when each feature is shuffled. It describes this model on this test set; it does not establish medical causation.

In [5]:
from sklearn.inspection import permutation_importance
importance = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring='roc_auc')
ranked = pd.DataFrame({'Feature': X.columns, 'Importance': importance.importances_mean}).sort_values('Importance', ascending=False).head(10)
print(ranked.to_string(index=False))
plt.figure(figsize=(8, 5))
sns.barplot(data=ranked, x='Importance', y='Feature', color='#4C78A8')
plt.title('Top permutation importances')
plt.show()

                Feature  Importance\n          worst texture    0.0280\n        worst concavity    0.0221\n     mean concave points    0.0159\n        worst perimeter    0.0138\n         worst compactness  0.0101\n

## 6. Conclusion and responsible-use notes

The model achieves strong held-out discrimination on this dataset, but it is **not ready for clinical use**. A real deployment would require external validation on contemporary and representative populations, calibration, fairness analysis, clinician oversight, privacy protections, and regulatory review. The next technical experiment would be threshold tuning based on a clearly defined clinical cost/benefit framework.